In [1]:
# Installation rapide
!pip install polars s3fs boto3 pyarrow pandas openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# Chargement dans la base concerné

In [12]:
import boto3
import io
import pandas as pd
from io import BytesIO

# Connexion à MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/DONNEES 2015/012015.xlsx"

# Télécharger le fichier Excel depuis S3
response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(response['Body'].read())

# Charger le fichier Excel
excel_file = pd.ExcelFile(bytes_io)

# Afficher les feuilles disponibles
print("Feuilles disponibles dans le fichier Excel :")
print(excel_file.sheet_names)

Feuilles disponibles dans le fichier Excel :
['Donnée1-012015', 'Donnée2-012015']


## Suppression de l'entête du fichier

In [13]:
import pandas as pd
from io import BytesIO

# 1. Lire depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
data_bytes = obj['Body'].read()

# Charger dans un buffer mémoire pour réutilisation
bio = BytesIO(data_bytes)

# 2. Lire la 2e feuille sans entête pour identifier la bonne ligne d'entête
data_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')

# Supprimer la dernière ligne (souvent vide ou non utile)
data_tmp = data_tmp.iloc[:-1].reset_index(drop=True)

# 3. Trouver dynamiquement la ligne contenant "matricule"
header_row = None
for i, row in data_tmp.iterrows():
    row_str = row.astype(str).str.lower().str.strip().tolist()
    if any("matricule" in cell for cell in row_str):  # Comparaison insensible à la casse
        header_row = i
        break

if header_row is None:
    raise ValueError("Aucune entête avec 'matricule' détectée !")

# 4. Relire depuis le début du fichier avec la bonne ligne d'entête
bio.seek(0)
data = pd.read_excel(bio, sheet_name=1, header=header_row, engine='openpyxl')

In [17]:
data.head()

,MATRICULE||CODE_ORGANISME,124,125,126,171,190,195,200,201,202,...,491,492,494,495,496,497,498,510,780,BRUT MONTANT
0,011242X15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34560
1,012856Q15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42400
2,013454N15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45600
3,027861L25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,800000
4,047690HBQ,513605.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,842646


# Fusion (On va segmenter pour faire la fusion de deux années)

## Fusion 2015 et 2016

In [14]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/", "Solde/DONNEES 2016/"]

In [15]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/"]

In [16]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")

nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


🔍 Fichiers détectés pour 12 périodes.
⚠️ Attention : seulement 12 mois détectés (au lieu de 24).
🗂️ Mois détectés : ['012015', '022015', '032015', '042015', '052015', '062015', '072015', '082015', '092015', '102015', '112015', '122015']


In [17]:
resultat_final = []

for mois, fichiers in fichiers_par_mois.items():
    for fichier in fichiers:
        try:
            # Lecture initiale sans en-tête
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj['Body'].read()
            bio = BytesIO(data_bytes)

            df_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')
            df_tmp.dropna(how='all', inplace=True)

            # Trouver une ligne d'en-tête contenant "matricule" (flexible)
            header_row_index = None
            for i, row in df_tmp.iterrows():
                row_clean = row.astype(str).str.lower().str.strip()
                if any("matricule" in cell for cell in row_clean):
                    header_row_index = i
                    break

            if header_row_index is None:
                print(f"⚠️ Pas d'en-tête trouvée dans {fichier}")
                continue

            # Relire à partir de l’en-tête trouvée
            bio.seek(0)
            df = pd.read_excel(bio, sheet_name=1, skiprows=header_row_index, engine='openpyxl')
            df["MOIS_COLLECTE"] = mois
            df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y")
            resultat_final.append(df)

            print(f"✅ Fichier ajouté : {fichier} ({len(df)} lignes)")

        except Exception as e:
            print(f"❌ Erreur pour {fichier} : {e}")


✅ Fichier ajouté : Solde/DONNEES 2015/012015.xlsx (184577 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/022015.xlsx (184805 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/032015.xlsx (185379 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/042015.xlsx (186893 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/052015.xlsx (188494 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/062015.xlsx (189611 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/072015.xlsx (190491 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/082015.xlsx (191621 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/092015.xlsx (192027 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/102015.xlsx (193957 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/112015.xlsx (194884 lignes)
✅ Fichier ajouté : Solde/DONNEES 2015/122015.xlsx (196097 lignes)


In [20]:
import boto3
import io
import pandas as pd

if resultat_final:
    # 1. Fusion
    panel = pd.concat(resultat_final, ignore_index=True)
    panel.dropna(how="all", inplace=True)
    panel = panel.loc[:, ~panel.columns.str.contains("^Unnamed")]

    # 2. Conversion objet → string
    for col in panel.columns:
        if panel[col].dtype == "object":
            panel[col] = panel[col].astype(str)

    print("✅ Fusion complète.")
    print(f"📊 Lignes : {len(panel)}")
    print(panel.head())

    # 3. Buffer Parquet
    buffer = io.BytesIO()
    panel.to_parquet(buffer, index=False)
    buffer.seek(0)

    # 4. Connexion MinIO
    s3_client = boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False   # HTTP, pas TLS : OK si réseau interne
    )

    # 5. Upload
    s3_client.put_object(
        Bucket="admindataanstat",
        Key="Solde/panel_solde_df_2015_16.parquet",
        Body=buffer.getvalue()
    )
    print("✅ Upload vers MinIO terminé.")

else:
    print("❌ Aucun fichier chargé.")


✅ Fusion complète.
📊 Lignes : 2278836
  MATRICULE||CODE_ORGANISME       124  125  126  171  190  195  200  201  202  \
0                 011242X15       NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
1                 012856Q15       NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2                 013454N15       NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
3                 027861L25       NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
4                 047690HBQ  513605.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

   ...  451  MONTANT BRUT  239  242  258  217  Étiquettes de colonnes  232  \
0  ...  NaN           NaN  NaN  NaN  NaN  NaN                     NaN  NaN   
1  ...  NaN           NaN  NaN  NaN  NaN  NaN                     NaN  NaN   
2  ...  NaN           NaN  NaN  NaN  NaN  NaN                     NaN  NaN   
3  ...  NaN           NaN  NaN  NaN  NaN  NaN                     NaN  NaN   
4  ...  NaN           NaN  NaN  NaN  NaN  NaN                     NaN  NaN   

   520

### Fusion 2017 et 2018 

In [21]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2017/", "Solde/DONNEES 2018/"]

In [22]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")
nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


🔍 Fichiers détectés pour 24 périodes.
✅ Tous les 24 mois ont bien été détectés.


In [ ]:
resultat_final = []

for mois, fichiers in fichiers_par_mois.items():
    for fichier in fichiers:
        try:
            # Lecture initiale sans en-tête
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj['Body'].read()
            bio = BytesIO(data_bytes)

            df_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')
            df_tmp.dropna(how='all', inplace=True)

            # Trouver une ligne d'en-tête contenant "matricule" (flexible)
            header_row_index = None
            for i, row in df_tmp.iterrows():
                row_clean = row.astype(str).str.lower().str.strip()
                if any("matricule" in cell for cell in row_clean):
                    header_row_index = i
                    break

            if header_row_index is None:
                print(f"⚠️ Pas d'en-tête trouvée dans {fichier}")
                continue

            # Relire à partir de l’en-tête trouvée
            bio.seek(0)
            df = pd.read_excel(bio, sheet_name=1, skiprows=header_row_index, engine='openpyxl')
            df["MOIS_COLLECTE"] = mois
            df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y")
            resultat_final.append(df)

            print(f"✅ Fichier ajouté : {fichier} ({len(df)} lignes)")

        except Exception as e:
            print(f"❌ Erreur pour {fichier} : {e}")


✅ Fichier ajouté : Solde/DONNEES 2017/012017.xlsx (202571 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/022017.xlsx (202378 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/032017.xlsx (203356 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/042017.xlsx (203902 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/052017.xlsx (204982 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/062017.xlsx (205709 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/072017.xlsx (206995 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/082017.xlsx (209192 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/092017.xlsx (208234 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/102017.xlsx (208151 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/112017.xlsx (209503 lignes)
✅ Fichier ajouté : Solde/DONNEES 2017/122017.xlsx (210645 lignes)
✅ Fichier ajouté : Solde/DONNEES 2018/012018.xlsx (211830 lignes)
✅ Fichier ajouté : Solde/DONNEES 2018/022018.xlsx (214207 lignes)
✅ Fichier ajouté : Solde/DONNEES 2018/032018.xlsx (214674 lignes)
✅ Fichier 

In [ ]:
import boto3
import io

if resultat_final:
    panel_solde_df_2017_18 = pd.concat(resultat_final, ignore_index=True)

    # Supprimer les lignes entièrement vides (optionnel, sécurité)
    panel_solde_df_2017_18.dropna(how='all', inplace=True)

    print("✅ Fusion complète en format panel.")
    print(f"📊 Nombre total de lignes : {len(panel_solde_df_2017_18)}")
    print(panel_solde_df_2017_18.head())

    # Sauvegarder dans un buffer mémoire au format parquet
    buffer = io.BytesIO()
    panel_solde_df_2017_18.to_parquet(buffer, index=False)
    buffer.seek(0)

    # Initialiser client S3
    #s3 = boto3.client('s3')
    
    # 4. Connexion MinIO
    s3_client = boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False   # HTTP, pas TLS : OK si réseau interne
    )

    # Envoyer le buffer dans le bucket S3
    s3.put_object(
        Bucket="admindataanstat",
        Key="Solde/panel_solde_df_2017_18.parquet",
        Body=buffer.getvalue()
    )

    print("✅ Fichier Parquet sauvegardé sur S3 : Solde/panel_solde_df_2017_18.parquet")

else:
    print("❌ Aucun fichier n'a été chargé correctement.")


# Fusion 2019 et 2020

In [ ]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2019/", "Solde/DONNEES 2020/"]

In [ ]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")

nb_mois = len(fichiers_par_mois)
if nb_mois == 24:
    print("✅ Tous les 24 mois ont bien été détectés.")
else:
    print(f"⚠️ Attention : seulement {nb_mois} mois détectés (au lieu de 24).")
    print("🗂️ Mois détectés :", sorted(fichiers_par_mois.keys()))


In [ ]:
resultat_final = []

for mois, fichiers in fichiers_par_mois.items():
    for fichier in fichiers:
        try:
            # Lecture initiale sans en-tête
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj['Body'].read()
            bio = BytesIO(data_bytes)

            df_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')
            df_tmp.dropna(how='all', inplace=True)

            # Trouver une ligne d'en-tête contenant "matricule" (flexible)
            header_row_index = None
            for i, row in df_tmp.iterrows():
                row_clean = row.astype(str).str.lower().str.strip()
                if any("matricule" in cell for cell in row_clean):
                    header_row_index = i
                    break

            if header_row_index is None:
                print(f"⚠️ Pas d'en-tête trouvée dans {fichier}")
                continue

            # Relire à partir de l’en-tête trouvée
            bio.seek(0)
            df = pd.read_excel(bio, sheet_name=1, skiprows=header_row_index, engine='openpyxl')
            df["MOIS_COLLECTE"] = mois
            df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y")
            resultat_final.append(df)

            print(f"✅ Fichier ajouté : {fichier} ({len(df)} lignes)")

        except Exception as e:
            print(f"❌ Erreur pour {fichier} : {e}")


In [ ]:
import boto3
import io

if resultat_final:
    panel_solde_df_2019_20 = pd.concat(resultat_final, ignore_index=True)

    # Supprimer les lignes entièrement vides (optionnel, sécurité)
    panel_solde_df_2019_20.dropna(how='all', inplace=True)

    print("✅ Fusion complète en format panel.")
    print(f"📊 Nombre total de lignes : {len(panel_solde_df_2019_20)}")
    print(panel_solde_df_2019_20.head())

    # Sauvegarder dans un buffer mémoire au format parquet
    buffer = io.BytesIO()
    panel_solde_df_2019_20.to_parquet(buffer, index=False)
    buffer.seek(0)

    # 4. Connexion MinIO
    s3_client = boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False   # HTTP, pas TLS : OK si réseau interne
    )


    # Envoyer le buffer dans le bucket S3
    s3.put_object(
        Bucket="admindataanstat",
        Key="Solde/panel_solde_df_2019_20.parquet",
        Body=buffer.getvalue()
    )

    print("✅ Fichier Parquet sauvegardé sur S3 : Solde/panel_solde_df_2019_20.parquet")

else:
    print("❌ Aucun fichier n'a été chargé correctement.")


In [ ]:
# ========== 4. Création du DataFrame panel ==========
if resultat_final:
    panel_solde_df_2015_16 = pd.concat(resultat_final, ignore_index=True)

    # Supprimer les lignes entièrement vides (optionnel, sécurité)
    panel_solde_df_2015_16.dropna(how='all', inplace=True)

    print("✅ Fusion complète en format panel.")
    print(f"📊 Nombre total de lignes : {len(panel_solde_df_2)}")
    print(panel_solde_df_2.head())

    # Sauvegarder dans le bucket S3
    Bucket="admindataanstat",
    Key="Solde/panel_solde_df_2015_16.parquet",
    Body=buffer
))
print("✅ Fichier Parquet sauvegardé sur S3 : Solde/panel_solde_df_2015_16.parquet")
else:
    print("❌ Aucun fichier n'a été chargé correctement.")

# Enregistrement de la base sous format CSV

In [ ]:
import io

# Convertir en CSV
csv_buffer = io.StringIO()
panel_solde_df.to_csv(csv_buffer, index=False, encoding="utf-8-sig")

# Sauvegarder dans le bucket S3
s3_client.put_object(
    Bucket="admindataanstat",
    Key="Solde/panel_solde_df_2.csv",
    Body=csv_buffer.getvalue()
)

print("✅ Ta base a été sauvegardée dans le bucket S3 : Solde/panel_solde_df_2.csv")